In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tabulate import tabulate
import os

path = "img/Semana 12"

def print_tabulate(df:pd.DataFrame)-> pd.DataFrame:
    print(tabulate(df, headers= df.columns, tablefmt='orgtbl'))

df = pd.read_csv("csv/aprovCreditos.csv", index_col=False)
os.makedirs(path, exist_ok=True)

df_sample = df[["Approved_Declined_Amount", "Disbursed_Shipped_Amount", "Program"]].sample(n=5000, random_state=42)

def get_cmap(n, name="hsv"):
    return plt.colormaps.get_cmap(name)

def euclidean_distance(p_1, p_2):
    return np.sqrt(np.sum((p_2 - p_1) ** 2))

def scatter_group_by(file_path, df, x_column, y_column, label_column):
    fig, ax = plt.subplots()
    labels = pd.unique(df[label_column])
    cmap = get_cmap(len(labels) + 1)
    for i, label in enumerate(labels):
        filter_df = df.query(f"{label_column} == '{label}'")
        ax.scatter(filter_df[x_column], filter_df[y_column], label=label)
    ax.legend()
    plt.set_cmap(cmap)
    plt.savefig(file_path)
    plt.close()

def calculate_means(points, labels, clusters):
    mean = []
    for k in range(clusters):
        m = np.mean(points[labels == k], axis=0)
        mean.append(m)
    return mean

def calculate_nearest_k(point, actual_means):
    distance = [euclidean_distance(mean, point) for mean in actual_means]
    return (point, np.argmin(distance))

def k_means(points, k):
    N = len(points)
    x = np.array(points)
    y = np.random.randint(0, k, N)
    dimensions = len(points[0])
    mean = np.zeros((k, dimensions)) 

    for t in range(15):
        actual_mean = calculate_means(points=x, labels=y, clusters=k)
        y = np.array([calculate_nearest_k(point=point, actual_means=actual_mean)[1] for point in x])

        df_points = pd.DataFrame(x, columns=['x', 'y'])
        df_points['label'] = np.char.mod('%d', y)
        df_mean = pd.DataFrame(actual_mean, columns=['x', 'y'])
        df_mean['label'] = ['centroid' for _ in range(len(actual_mean))]
        df_plot = pd.concat([df_points, df_mean])
        scatter_group_by(f"{path}/kmeans_{t}.png", df_plot, "x", "y", "label")

        if np.array_equal(np.array(actual_mean), mean):
            break
        mean = np.array(actual_mean).copy()
    return mean

list_t = [
    (np.array([row["Approved_Declined_Amount"], row["Disbursed_Shipped_Amount"]]), 
     row["Program"])
    for _, row in df_sample.iterrows()
]
points = [point for point, _ in list_t]

kn = k_means(points, 3)
print(f"Centroides finales: {kn}")

Centroides finales: [[1.80445454e+06 2.23806443e+06]
 [1.41092114e+09 1.45381131e+09]
 [3.03919167e+08 2.84612048e+08]]
